# NB01 — Per-cluster Pfam Coverage Extraction

**Environment:** BERDL JupyterHub (on-cluster Spark).

**Purpose:** For every gene cluster, assign a Pfam-based coverage tier by intersecting `kbase_ke_pangenome.interproscan_domains` (analysis='Pfam') with `kescience_pdb.pdb_pfam`.

**Tier assignment:**
- `no_pfam_annotation`: no Pfam annotation exists for this cluster
- `pfam_no_covered`: has Pfam(s), none in PDB
- `pfam_partial_covered`: has multiple Pfams, some in PDB
- `pfam_all_covered`: all Pfams have PDB structure

**Why IPS-Pfam, not `bakta_pfam_domains`:** bakta silently drops half of pangenome Pfams (10,798 distinct vs. 20,273 in IPS). Per-cluster annotation rate: bakta 7.7% vs IPS 72.3%. See RESEARCH_PLAN.md v1 revision.

**Output:** all downstream aggregates (biome × tier × core matrix, biome summary, top uncovered Pfams, species/genome biome maps) are written by the same script — see `scripts/01_extract_and_stratify.py`.

In [ ]:
# The extraction was executed via scripts/01_extract_and_stratify.py to write local CSVs
# (Spark Connect writes go to worker filesystem, not local — bypass via toPandas().to_csv()).
#
# To re-run:
#   python scripts/01_extract_and_stratify.py
#
# The pipeline sections below mirror what the script does, and are runnable as-is inside
# a JupyterHub kernel where writes to the shared filesystem work directly.

from berdl_notebook_utils.setup_spark_session import get_spark_session
from pyspark.sql.functions import regexp_replace, col, when, lit, count, countDistinct, sum as spark_sum

spark = get_spark_session()

## PDB Pfam universe

In [ ]:
pdb_pfams = spark.table("kescience_pdb.pdb_pfam").select("pfam_id").distinct()
n_pdb_pfams = pdb_pfams.count()
print(f'Distinct Pfams with any PDB structure: {n_pdb_pfams:,}')  # ~12,053

## Bakta Pfam universe (from IPS)

In [ ]:
ips_pfam = (spark.table('kbase_ke_pangenome.interproscan_domains')
    .filter("analysis = 'Pfam'")
    .select('gene_cluster_id',
            regexp_replace(col('signature_acc'), r'\.\d+$', '').alias('pfam_id'))
    .distinct())

n_bakta_pfams = ips_pfam.select('pfam_id').distinct().count()
print(f'Distinct Pfams in pangenome (via IPS): {n_bakta_pfams:,}')  # ~20,273

## Per-cluster tier

In [ ]:
per_cluster_pfam = ips_pfam.groupBy('gene_cluster_id').agg(
    countDistinct('pfam_id').alias('n_pfam'))

bakta_covered = (ips_pfam.join(pdb_pfams, on='pfam_id', how='inner')
    .groupBy('gene_cluster_id')
    .agg(countDistinct('pfam_id').alias('n_covered_pfam')))

per_cluster_tier = (per_cluster_pfam
    .join(bakta_covered, on='gene_cluster_id', how='left')
    .withColumn('n_covered_pfam', when(col('n_covered_pfam').isNull(), 0).otherwise(col('n_covered_pfam')))
    .withColumn('pfam_tier',
        when(col('n_covered_pfam') == 0, lit('pfam_no_covered'))
        .when(col('n_covered_pfam') == col('n_pfam'), lit('pfam_all_covered'))
        .otherwise(lit('pfam_partial_covered'))))

print('=== Tier distribution among clusters with any Pfam ===')
per_cluster_tier.groupBy('pfam_tier').count().orderBy('pfam_tier').show()

## Coverage of the pangenome (with vs. without Pfam)

In [ ]:
total = spark.table('kbase_ke_pangenome.gene_cluster').select('gene_cluster_id').distinct().count()
with_pfam = per_cluster_tier.count()
print(f'Total gene clusters: {total:,}')
print(f'With any IPS-Pfam annotation: {with_pfam:,} ({100*with_pfam/total:.1f}%)')
print(f'With no IPS-Pfam annotation: {total - with_pfam:,} ({100*(total-with_pfam)/total:.1f}%)')